# Phase 2: Exploratory Data Analysis (EDA) — DENTEX Abnormal Tooth Detection

> **DISCLAIMER:** This project is an **educational and research prototype only**. It is **NOT** a clinical diagnostic or treatment system. All visualizations and inferences are solely for university research and engineering evaluation.

### Primary Research Question:
*Does a detector + CNN-confirmation pipeline improve macro-F1 over the YOLO detector alone across 4 target classes (`caries`, `deep_caries`, `periapical_lesion`, `impacted_tooth`), while staying fast enough for interactive use?*

---
### Objectives in this Notebook:
1. Validate DENTEX COCO-style annotation schema.
2. Check and quantify **class imbalance** across the 4 diagnostic classes.
3. Analyze **image resolutions and aspect ratios** across dental panoramic radiographs.
4. Inspect **bounding box scales and aspect ratios** (informing YOLO anchor sizes and CNN crop padding).
5. Verify **image-level split isolation** to prevent data leakage.

In [ ]:
import sys
from pathlib import Path
# Ensure project root is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

from dentistry.config import load_config
from dentistry.dataset import parse_coco_annotations, validate_coco, DEFAULT_CLASSES, _normalise_category_name

cfg = load_config(project_root / "configs/default.yaml")
print("Loaded Project Configuration:", cfg["project"]["name"])
print("Target Classes:", cfg["classes"])

## 1. Annotation Loading & Dataset Schema Validation

In [ ]:
# Determine annotation and image paths (uses sample data if full DENTEX is not yet downloaded)
raw_ann = project_root / "data/DENTEX/annotations/abnormal_teeth.json"
sample_ann = project_root / "data/DENTEX_sample/annotations/sample_abnormal_teeth.json"

if raw_ann.exists():
    ann_path = raw_ann
    img_root = project_root / "data/DENTEX/images"
    print(f"Using real DENTEX annotations from: {ann_path}")
elif sample_ann.exists():
    ann_path = sample_ann
    img_root = project_root / "data/DENTEX_sample/images"
    print(f"Using sample dataset annotations from: {ann_path}")
else:
    # Generate sample dataset on the fly
    from scripts.generate_sample_data import generate_panoramic_sample
    res = generate_panoramic_sample(project_root / "data/DENTEX_sample", num_images=12)
    ann_path = Path(res["annotations_path"])
    img_root = Path(res["images_dir"])
    print(f"Generated and loaded synthetic sample data from: {ann_path}")

parsed = parse_coco_annotations(ann_path, img_root)
validation = validate_coco(parsed)

print(f"\nTotal Registered Images: {validation['image_count']}")
print(f"Total Bounding Boxes: {validation['annotation_count']}")
print(f"Invalid BBoxes: {len(validation['invalid_bboxes'])}")
print(f"Missing Files: {len(validation['missing_image_files'])}")

## 2. Class Distribution & Imbalance Analysis
In dental panoramic X-rays, superficial caries are far more common than deep caries or periapical lesions. Examining class frequencies is essential for configuring weighted cross-entropy loss or balanced samplers.

In [ ]:
ann_rows = []
for a in parsed["annotations"]:
    if a.get("bbox_valid") and a.get("bbox"):
        cname = _normalise_category_name(a.get("category_name", ""))
        bx, by, bw, bh = a["bbox"]
        ann_rows.append({
            "ann_id": a["id"],
            "image_id": a["image_id"],
            "category": cname,
            "bbox_w": bw,
            "bbox_h": bh,
            "bbox_area": bw * bh,
            "aspect_ratio": bw / bh if bh > 0 else np.nan
        })
df_anns = pd.DataFrame(ann_rows)

class_counts = df_anns["category"].value_counts()
print("Class Counts:")
print(class_counts)

fig, ax = plt.subplots(figsize=(7, 4))
colors = {"caries": "#e74c3c", "deep_caries": "#c0392b", "periapical_lesion": "#e67e22", "impacted_tooth": "#2980b9"}
bar_colors = [colors.get(c, "#7f8c8d") for c in class_counts.index]
bars = ax.bar(class_counts.index, class_counts.values, color=bar_colors, edgecolor="black", linewidth=0.6)
for b in bars:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width() / 2, h + 0.3, f"{int(h)}", ha="center", fontweight="bold")
ax.set_title("Class Distribution in Dataset", fontsize=12, fontweight="bold")
ax.set_ylabel("Number of Annotations")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Panoramic Image Resolutions and Aspect Ratios
Dental panoramic radiographs typically feature a wide aspect ratio (~2:1). Understanding image dimensions helps select the proper input resolution (`imgsz = 640` for YOLO with letterboxing).

In [ ]:
img_rows = []
for im in parsed["images"]:
    w, h = im.get("width"), im.get("height")
    img_rows.append({
        "image_id": im["id"],
        "width": w,
        "height": h,
        "aspect_ratio": w / h if (w and h) else np.nan
    })
df_imgs = pd.DataFrame(img_rows)
print(df_imgs.describe())

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df_imgs["width"], df_imgs["height"], color="#2c3e50", alpha=0.7, s=50)
ax.set_title("Image Width vs Height", fontsize=11, fontweight="bold")
ax.set_xlabel("Width (px)")
ax.set_ylabel("Height (px)")
ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

## 4. Bounding Box Scale Analysis (Crop Context Sizing)
Because caries and periapical lesions represent small fractions of the total panoramic image (< 2% of pixels), extracting crops with context padding (e.g., 10%) is critical so the CNN can observe adjacent enamel and bone margins.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.hist(df_anns["bbox_w"], bins=15, color="#3498db", edgecolor="black", alpha=0.7)
ax1.set_title("Bounding Box Width (px)", fontweight="bold")
ax1.grid(True, linestyle=":", alpha=0.5)

ax2.hist(df_anns["aspect_ratio"], bins=15, color="#9b59b6", edgecolor="black", alpha=0.7)
ax2.set_title("BBox Aspect Ratio (w / h)", fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Visualizing Sample Panoramic X-rays with Ground Truth

In [ ]:
sample_img_info = parsed["images"][0]
img_path = sample_img_info.get("path") or str(img_root / sample_img_info["file_name"])

if Path(img_path).exists():
    sample_img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(sample_img)
    
    for a in parsed["annotations"]:
        if a["image_id"] == sample_img_info["id"] and a.get("bbox_valid") and a.get("bbox"):
            cname = _normalise_category_name(a.get("category_name", ""))
            bx, by, bw, bh = a["bbox"]
            c = colors.get(cname, "yellow")
            draw.rectangle([bx, by, bx + bw, by + bh], outline=c, width=4)
            draw.text((bx + 2, max(0, by - 12)), cname, fill=c)
    
    plt.figure(figsize=(10, 5))
    plt.imshow(sample_img)
    plt.title(f"Ground Truth Annotations: {sample_img_info['file_name']}", fontweight="bold")
    plt.axis("off")
    plt.show()
else:
    print(f"Sample image not found at {img_path}")

## 6. Key Takeaways & Risk Mitigation Strategies

1. **Patient / Image-Level Leakage Mitigation:**
   - We enforce strict partitioning by `image_id`. Teeth from the same panoramic radiograph never appear in both train and test.

2. **Severe Class Imbalance:**
   - Periapical lesions and deep caries have significantly fewer examples than superficial caries.
   - *Mitigation:* Class-weighted loss in the CNN classifier and focal loss in YOLO.

3. **Small Target Scales:**
   - Dental lesions occupy small spatial areas (< 2% of the panoramic canvas).
   - *Mitigation:* Cropping tooth regions with 10% bounding box context padding allows the downstream CNN to resolve delicate apical and marginal structures without spatial loss.